In [1]:
import json
import re
import pandas as pd

def create_rag_evaluation_dataset(input_json_file, output_csv_file):
    print(f"Đang đọc dữ liệu từ: {input_json_file}")
    
    # 1. Đọc file JSON
    with open(input_json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    rag_test_data = []

    # 2. Duyệt qua từng bài viết
    for doc in data:
        title = doc.get("title", "")
        url = doc.get("url", "")
        # ĐÂY CHÍNH LÀ ĐOẠN VĂN SẠCH DÙNG CHO FAISS VÀ MODEL ĐỌC
        context = doc.get("rule_cleaned_content", "") 
        qa_text = doc.get("generated_qa", "")

        # 3. Dùng Regex để bóc tách từng cặp Câu hỏi (CH) và Câu trả lời (ĐA)
        pairs = re.findall(
            r"CH\d+:\s*(.*?)\s*\nĐA\d+:\s*(.*?)(?=\nCH\d+:|$)",
            qa_text + "\n",
            re.DOTALL
        )

        # 4. Ghi từng cặp QA thành 1 dòng riêng biệt
        for q, a in pairs:
            q = q.strip()
            a = a.strip()

            if q and a:
                rag_test_data.append({
                    "Question": q,             # Câu hỏi để đưa vào hàm tìm kiếm
                    "Ground_Truth": a,         # Đáp án chuẩn (để tính điểm Exact Match, F1)
                    "Context": context,        # Văn bản gốc (để đưa vào FAISS database)
                    "Title": title,            # Metadata (để biết nguồn)
                    "URL": url                 # Metadata (để biết nguồn)
                })

    # 5. Xuất ra file CSV
    df = pd.DataFrame(rag_test_data)
    df.to_csv(output_csv_file, index=False, encoding="utf-8-sig")
    
    print(f"✅ Đã tạo thành công file: {output_csv_file}")
    print(f"🎯 Tổng số câu hỏi test tạo được: {len(df)}")
    
    # In thử 1 dòng để kiểm tra
    if len(df) > 0:
        print("\n--- MẪU DỮ LIỆU ĐÃ BÓC TÁCH ---")
        print(f"Câu hỏi : {df.iloc[0]['Question']}")
        print(f"Đáp án  : {df.iloc[0]['Ground_Truth']}")
        print(f"Context : {df.iloc[0]['Context'][:100]}...")

# Chạy hàm (Thay tên file của bạn vào đây)
create_rag_evaluation_dataset("/kaggle/input/notebooks/nguyenthingoclanuet/generate-data/uet_qa_dataset.json", "rag_test_dataset.csv")

Đang đọc dữ liệu từ: /kaggle/input/notebooks/nguyenthingoclanuet/generate-data/uet_qa_dataset.json
✅ Đã tạo thành công file: rag_test_dataset.csv
🎯 Tổng số câu hỏi test tạo được: 399

--- MẪU DỮ LIỆU ĐÃ BÓC TÁCH ---
Câu hỏi : Khi nào sinh viên thực hiện việc tự đánh giá kết quả rèn luyện học kỳ I năm học 2024-2025?
Đáp án  : Từ 8h00 ngày 03/03/2025 đến 8h00 ngày 07/03/2025.
Context : Thực hiện kế hoạch học tập năm học 2024-2025, phòng Công tác Sinh viên thông báo Kế hoạch tổ chức đá...


In [2]:
!pip install -q faiss-cpu sentence-transformers langchain-text-splitters langchain-community transformers accelerate bitsandbytes pandas rank-bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pan

In [3]:
import sys
import torch
import gc

# Dọn RAM
gc.collect()
torch.cuda.empty_cache()

sys.path.append('/kaggle/input/notebooks/nguyenthingoclanuet/rag-system/RAG')
from retriever import HybridRetriever
from generator import QAGenerator

# Load đúng thư mục chứa 3 file index
retriever = HybridRetriever()
retriever.load("/kaggle/input/notebooks/nguyenthingoclanuet/rag-system/my_uet_index") 

generator = QAGenerator()

query = "Trong kỳ tuyển dụng viên chức hành chính năm 2023 của Trường Đại học Công nghệ, Đại học Quốc gia Hà Nội, có bao nhiêu ứng viên được triệu tập tham dự thi Vòng 1?"
docs = retriever.search(query, top_k=1) # 500 câu ngắn thì chỉ cần top 2 là đủ
answer = generator.get_answer(query, docs)

print("\n" + "="*30)
print(f"CÂU HỎI: {query}")
print(f"ĐÁP ÁN: {answer}")
print("="*30)

Loading BGE-M3 Bi-encoder...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



CÂU HỎI: Trong kỳ tuyển dụng viên chức hành chính năm 2023 của Trường Đại học Công nghệ, Đại học Quốc gia Hà Nội, có bao nhiêu ứng viên được triệu tập tham dự thi Vòng 1?
ĐÁP ÁN: 20 người


In [4]:
# import os
# import re
# import json
# import string
# import torch
# import pandas as pd
# import numpy as np
# from tqdm import tqdm
# from collections import Counter
# # from retriever import HybridRetriever
# # from generator import QAGenerator

# # =========================================================
# # UTILS: CHUẨN HÓA VĂN BẢN
# # =========================================================
# def normalize_answer(s):
#     """Lấy chữ thường, bỏ dấu câu, bỏ khoảng trắng thừa"""
#     def remove_articles(text):
#         return re.sub(r'\b(a|an|the)\b', ' ', text)

#     def white_space_fix(text):
#         return ' '.join(text.split())

#     def remove_punc(text):
#         exclude = set(string.punctuation)
#         return ''.join(ch for ch in text if ch not in exclude)

#     def lower(text):
#         return text.lower()

#     return white_space_fix(remove_punc(lower(str(s))))

# # =========================================================
# # METRICS: EM, F1, PRECISION, RECALL (TOKEN LEVEL)
# # =========================================================
# def calc_f1_precision_recall(prediction, ground_truth):
#     prediction_tokens = normalize_answer(prediction).split()
#     ground_truth_tokens = normalize_answer(ground_truth).split()
    
#     if len(prediction_tokens) == 0 or len(ground_truth_tokens) == 0:
#         return int(prediction_tokens == ground_truth_tokens), 0, 0, 0

#     common = Counter(prediction_tokens) & Counter(ground_truth_tokens)
#     num_same = sum(common.values())
    
#     if num_same == 0:
#         return 0, 0, 0, 0
    
#     precision = 1.0 * num_same / len(prediction_tokens)
#     recall = 1.0 * num_same / len(ground_truth_tokens)
#     f1 = (2 * precision * recall) / (precision + recall)
    
#     return int(normalize_answer(prediction) == normalize_answer(ground_truth)), f1, precision, recall

# # =========================================================
# # MAIN EVALUATION
# # =========================================================
# def run_evaluation(csv_path, index_dir, num_samples=100):
#     print("--- ĐANG KHỞI TẠO HỆ THỐNG RAG ---")
#     # retriever = HybridRetriever()
#     # retriever.load(index_dir)
#     # generator = QAGenerator()

#     # Load dữ liệu test
#     df = pd.read_csv(csv_path, encoding="utf-8-sig")
#     if num_samples < len(df):
#         df = df.sample(num_samples, random_state=42)

#     results = []
    
#     print(f"--- ĐANG ĐÁNH GIÁ TRÊN {len(df)} MẪU ---")
#     for _, row in tqdm(df.iterrows(), total=len(df)):
#         question = row['Question']
#         ground_truth = row['Answer']
        
#         # 1. Retrieval
#         retrieved_docs = retriever.search(question, top_k=3)
#         context_text = " ".join([d['text'] for d in retrieved_docs])
        
#         # Kiểm tra xem Ground Truth có xuất hiện trong Context không (Retrieval Recall)
#         retrieval_success = 1 if normalize_answer(ground_truth) in normalize_answer(context_text) else 0
        
#         # 2. Generation
#         try:
#             prediction = generator.get_answer(question, retrieved_docs)
#         except Exception as e:
#             print(f"Lỗi khi generate: {e}")
#             prediction = ""

#         # 3. Calculate Metrics
#         em, f1, prec, rec = calc_f1_precision_recall(prediction, ground_truth)
        
#         results.append({
#             "Question": question,
#             "Ground Truth": ground_truth,
#             "Prediction": prediction,
#             "EM": em,
#             "F1": f1,
#             "Precision": prec,
#             "Recall": rec,
#             "Retrieval_Success": retrieval_success
#         })

#         # Giải phóng bộ nhớ GPU tránh OOM trên Kaggle
#         torch.cuda.empty_cache()

#     # Tạo DataFrame kết quả
#     res_df = pd.DataFrame(results)
    
#     # Tính trung bình
#     print("\n" + "="*40)
#     print("KẾT QUẢ ĐÁNH GIÁ CHI TIẾT")
#     print("="*40)
#     print(f"Exact Match (EM):      {res_df['EM'].mean():.4f}")
#     print(f"F1-Score:              {res_df['F1'].mean():.4f}")
#     print(f"Precision:             {res_df['Precision'].mean():.4f}")
#     print(f"Recall (Generation):   {res_df['Recall'].mean():.4f}")
#     print(f"Retrieval Recall@3:    {res_df['Retrieval_Success'].mean():.4f}")
#     print("="*40)
    
#     # Lưu file report
#     report_path = "/kaggle/working/evaluation_report.csv"
#     res_df.to_csv(report_path, index=False, encoding="utf-8-sig")
#     print(f"Đã lưu báo cáo chi tiết vào: {report_path}")

# if __name__ == "__main__":
#     CSV_TEST = "/kaggle/input/notebooks/nguyenthingoclanuet/generate-data/uet_qa_dataset_with_id.csv"
#     INDEX_DIR = "/kaggle/input/notebooks/nguyenthingoclanuet/rag-system/my_uet_index"
    
#     # Bạn có thể đổi num_samples thành len(df) để test hết 500 câu
#     run_evaluation(CSV_TEST, INDEX_DIR, num_samples=50)

In [5]:
import os
import re
import json
import string
import torch
import gc
import pandas as pd
import numpy as np
from tqdm import tqdm
from collections import Counter

# Note: Ensure these classes are imported or defined in your environment
# from retriever import HybridRetriever
# from generator import QAGenerator

# =========================================================
# UTILS: TEXT NORMALIZATION
# =========================================================
def normalize_answer(s):
    """Lower case, remove punctuation, remove articles and extra whitespace."""
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)

    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_punc(lower(str(s))))

# =========================================================
# METRICS: EM, F1, PRECISION, RECALL
# =========================================================
def calc_f1_precision_recall(prediction, ground_truth):
    if not prediction or prediction.startswith("ERROR"):
        return 0, 0, 0, 0
        
    prediction_tokens = normalize_answer(prediction).split()
    ground_truth_tokens = normalize_answer(ground_truth).split()
    
    if len(prediction_tokens) == 0 or len(ground_truth_tokens) == 0:
        return int(prediction_tokens == ground_truth_tokens), 0, 0, 0

    common = Counter(prediction_tokens) & Counter(ground_truth_tokens)
    num_same = sum(common.values())
    
    if num_same == 0:
        return 0, 0, 0, 0
    
    precision = 1.0 * num_same / len(prediction_tokens)
    recall = 1.0 * num_same / len(ground_truth_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    
    em = int(normalize_answer(prediction) == normalize_answer(ground_truth))
    return em, f1, precision, recall

# =========================================================
# MAIN EVALUATION SCRIPT
# =========================================================
def run_evaluation(csv_path, index_dir, num_samples=100, save_every=10):
    print("--- INITIALIZING RAG SYSTEM ---")
    
    # Initialize your models here
    # retriever = HybridRetriever()
    # retriever.load(index_dir)
    # generator = QAGenerator() 
    # Ensure generator model is in .eval() mode and half precision if possible
    # generator.model.half() 

    # Load test data
    df = pd.read_csv(csv_path, encoding="utf-8-sig")
    if num_samples < len(df):
        df = df.sample(num_samples, random_state=42)

    results = []
    report_path = "/kaggle/working/evaluation_report.csv"
    
    print(f"--- EVALUATING {len(df)} SAMPLES ---")
    
    # Disable gradient calculations globally for evaluation
    with torch.no_grad():
        for i, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df))):
            question = row['Question']
            ground_truth = row['Answer']
            
            prediction = ""
            retrieval_success = 0
            
            try:
                # 1. Retrieval
                retrieved_docs = retriever.search(question, top_k=3)
                context_text = " ".join([d['text'] for d in retrieved_docs])
                
                # Retrieval Recall check
                retrieval_success = 1 if normalize_answer(ground_truth) in normalize_answer(context_text) else 0
                
                # 2. Generation
                prediction = generator.get_answer(question, retrieved_docs)
                
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print(f"\n[WARNING] CUDA OOM at index {i}. Clearing cache and skipping...")
                    prediction = "ERROR_CUDA_OOM"
                    # Force clean up
                    if hasattr(torch.cuda, 'empty_cache'):
                        torch.cuda.empty_cache()
                    gc.collect()
                else:
                    print(f"\n[ERROR] Runtime Error: {e}")
                    prediction = f"ERROR_RUNTIME: {str(e)[:50]}"
            except Exception as e:
                print(f"\n[ERROR] Unexpected Error: {e}")
                prediction = f"ERROR_GENERAL: {str(e)[:50]}"

            # 3. Calculate Metrics
            em, f1, prec, rec = calc_f1_precision_recall(prediction, ground_truth)
            
            results.append({
                "Question": question,
                "Ground Truth": ground_truth,
                "Prediction": prediction,
                "EM": em,
                "F1": f1,
                "Precision": prec,
                "Recall": rec,
                "Retrieval_Success": retrieval_success
            })

            # 4. MEMORY MANAGEMENT
            # Explicitly delete large objects to help Garbage Collector
            if 'retrieved_docs' in locals(): del retrieved_docs
            
            # Periodically clear GPU cache and run Python GC
            if (i + 1) % 5 == 0:
                torch.cuda.empty_cache()
                gc.collect()

            # 5. PROGRESSIVE SAVING
            if (i + 1) % save_every == 0:
                pd.DataFrame(results).to_csv(report_path, index=False, encoding="utf-8-sig")

    # Final Save
    res_df = pd.DataFrame(results)
    res_df.to_csv(report_path, index=False, encoding="utf-8-sig")
    
    # Final Statistics (filter out error rows for accurate metrics)
    valid_res = res_df[~res_df['Prediction'].str.contains("ERROR", na=False)]
    
    print("\n" + "="*40)
    print(f"FINAL REPORT ({len(valid_res)}/{len(df)} Successful)")
    print("="*40)
    if not valid_res.empty:
        print(f"Exact Match (EM):      {valid_res['EM'].mean():.4f}")
        print(f"F1-Score:              {valid_res['F1'].mean():.4f}")
        print(f"Precision:             {valid_res['Precision'].mean():.4f}")
        print(f"Recall (Gen):          {valid_res['Recall'].mean():.4f}")
        print(f"Retrieval Recall@3:    {valid_res['Retrieval_Success'].mean():.4f}")
    else:
        print("No valid results to calculate metrics.")
    print("="*40)
    print(f"Báo cáo chi tiết: {report_path}")

if __name__ == "__main__":
    # Settings
    CSV_TEST = "/kaggle/input/notebooks/nguyenthingoclanuet/generate-data/uet_qa_dataset_with_id.csv"
    INDEX_DIR = "/kaggle/input/notebooks/nguyenthingoclanuet/rag-system/my_uet_index"
    
    # For a full run of 500 questions, keep num_samples=500
    run_evaluation(CSV_TEST, INDEX_DIR, num_samples=500, save_every=20)

--- INITIALIZING RAG SYSTEM ---
--- EVALUATING 399 SAMPLES ---



  0%|          | 0/399 [00:00<?, ?it/s]


[WARNING] CUDA OOM at index 0. Clearing cache and skipping...



  4%|▍         | 16/399 [01:34<43:51,  6.87s/it]


[WARNING] CUDA OOM at index 16. Clearing cache and skipping...


...


[WARNING] CUDA OOM at index 380. Clearing cache and skipping...



 97%|█████████▋| 387/399 [36:29<01:30,  7.58s/it]


[WARNING] CUDA OOM at index 387. Clearing cache and skipping...



100%|██████████| 399/399 [37:48<00:00,  5.69s/it]


FINAL REPORT (331/399 Successful)
Exact Match (EM):      0.3082
F1-Score:              0.5875
Precision:             0.6978
Recall (Gen):          0.5606
Retrieval Recall@3:    0.6224
Báo cáo chi tiết: /kaggle/working/evaluation_report.csv


In [6]:
# !export PYTHONPATH=$PYTHONPATH:/kaggle/input/notebooks/nguyenthingoclanuet/rag-system/RAG && python /kaggle/input/notebooks/nguyenthingoclanuet/rag-system/RAG/evaluate.py